## Exploratory notebook that saves all valid DCE and mask paths for patients at their respective time points

From now on, I can work from the JSON file without having to filter invalid patients, dates, or series fodlers

In [24]:
import os
import zipfile
import json
from pathlib import Path
from datetime import datetime
from tqdm import tqdm


In [ ]:
SCRATCH_DATA_DIR = Path('/mnt/scratch/gerlac37/ISPY2/data_copy')
XLSX_REF = Path('/mnt/home/gerlac37/ISPY2/data/Multi-feature-MRI-NACT-Data.xlsx')

HOME_DATA_DIR = Path('/mnt/home/gerlac37/ISPY2')
VALID_OUTPUT_PATH = HOME_DATA_DIR / 'data/valid_patient_information.json'
INVALID_OUTPUT_PATH = HOME_DATA_DIR / 'data/invalid_patient_information.json'

In [26]:
patient_folders = [p.name for p in SCRATCH_DATA_DIR.iterdir() if p.is_dir()]

valid_patient_infos = {}
invalid_patient_dirs = {}

for patient_folder in tqdm(patient_folders, desc="Cleaning Patient Information To JSON"):
    patient_path = SCRATCH_DATA_DIR / patient_folder
    try:
        patient_id = int(patient_folder.split('-')[-1])
    except ValueError:
        invalid_patient_dirs[patient_folder] = {
            "error": "cannot parse patient_id from folder name",
            "path": str(patient_path)
        }
        continue

    # Prepare containers
    valid_patient_infos[patient_id] = {
        'Patient Folder Path': str(patient_path),
        'Time Labels': []
    }
    bad_dates = []

    # Collect and sort only valid date‐folders
    date_folders = []
    for df in patient_path.iterdir():
        if not df.is_dir():
            bad_dates.append(str(df))
            continue
        try:
            # raises ValueError if format mismatch
            _ = datetime.strptime(df.name, '%m-%d-%Y')
        except ValueError:
            bad_dates.append(str(df))
            continue
        date_folders.append(df)
    # sort by actual date
    date_folders.sort(key=lambda d: datetime.strptime(d.name, '%m-%d-%Y'))

    # Process each date
    all_invalid_series = []
    for time_idx, date_folder_path in enumerate(date_folders):
        # find matching json & zip series IDs
        json_stems = {f.stem for f in date_folder_path.glob('*.json')}
        zip_stems  = {f.stem for f in date_folder_path.glob('*.zip')}
        common = json_stems & zip_stems

        # test zip validity
        valid_zips = []
        for sid in common:
            zp = date_folder_path / f"{sid}.zip"
            if zipfile.is_zipfile(zp):
                valid_zips.append(sid)
            else:
                all_invalid_series.append(sid)

        # classify into DCE vs SEG
        seg_ids = []
        dce_ids = []
        for sid in valid_zips:
            try:
                meta = json.loads((date_folder_path / f"{sid}.json").read_text())
            except ValueError:
                all_invalid_series.append(sid)
                continue
            mod  = (meta.get('Modality') or "").upper()
            desc = (meta.get('Series Description') or "").upper()
            if mod == 'SEG':
                seg_ids.append(sid)
            if 'DCE' in desc:
                dce_ids.append(sid)

        # skip bad dates
        if len(seg_ids) != 1 or len(dce_ids) != 1:
            bad_dates.append(str(date_folder_path))
            continue

        seg_id = seg_ids[0]
        dce_id = dce_ids[0]

        valid_patient_infos[patient_id]['Time Labels'].append({
            'Label':            f"T{time_idx}",
            'Date':             date_folder_path.name,
            'Date Folder Path': str(date_folder_path),
            'Mask Series ID':   seg_id,
            'Mask Zip Path':    str(date_folder_path / f"{seg_id}.zip"),
            'DCE Series ID':    dce_id,
            'DCE Zip Path':     str(date_folder_path / f"{dce_id}.zip"),
        })

    invalid_patient_dirs[patient_folder] = {
        'Dates':  bad_dates,
        'Series': all_invalid_series
    }

    if not valid_patient_infos[patient_id]['Time Labels']:
        del valid_patient_infos[patient_id]

with open(VALID_OUTPUT_PATH, "w") as file:
    json.dump(valid_patient_infos, file, indent=2)

with open(INVALID_OUTPUT_PATH,'w') as f:
    json.dump(invalid_patient_dirs, f, indent=2)


Cleaning Patient Information To JSON: 100%|██████████| 985/985 [04:24<00:00,  3.73it/s]
